In [65]:
import geopandas as gpd

roads = gpd.read_file("../data/gis/roads_final.geojson")
bridges = gpd.read_file("../data/gis/bridges_final.geojson")

print("Roads:", len(roads))
print("Bridges:", len(bridges))

print(roads.columns)
print(bridges.columns)

roads.head()


Roads: 62143
Bridges: 867
Index(['road_id', 'highway', 'geometry'], dtype='object')
Index(['bridge_id', 'geometry'], dtype='object')


,road_id,highway,geometry
0,road_0,residential,"LINESTRING (73.91666 18.45054, 73.91679 18.450..."
1,road_1,residential,"LINESTRING (73.9143 18.45015, 73.91431 18.4500..."
2,road_2,residential,"LINESTRING (73.91505 18.45049, 73.91523 18.449..."
3,road_3,trunk,"LINESTRING (73.91505 18.45049, 73.91502 18.450..."
4,road_4,trunk,"LINESTRING (73.91607 18.4507, 73.91577 18.4506..."


In [66]:
print("Original CRS:", roads.crs)

roads = roads.to_crs(epsg=32643)     # UTM zone for Pune region
bridges = bridges.to_crs(epsg=32643)

print("New CRS:", roads.crs)


Original CRS: EPSG:4326
New CRS: EPSG:32643


In [67]:
roads["highway"].value_counts().head(10)


highway
residential      31788
service          20579
tertiary          1973
footway           1773
primary           1205
living_street     1003
secondary          883
trunk              791
track              582
path               465
Name: count, dtype: int64

In [68]:
roads["length_m"] = roads.geometry.length

roads["length_m"].describe()


count    62143.000000
mean       140.572323
std        350.108843
min          0.523005
25%         40.211424
50%         80.128878
75%        159.681008
max      66931.301789
Name: length_m, dtype: float64

In [69]:
import numpy as np

# base traffic weights by road type
traffic_map = {
    "motorway": 5,
    "trunk": 4,
    "primary": 3,
    "secondary": 2.5,
    "tertiary": 2,
    "residential": 1.5,
    "service": 1,
    "living_street": 0.8,
    "footway": 0.3,
    "path": 0.2,
    "track": 0.5
}

roads["traffic_base"] = roads["highway"].map(traffic_map).fillna(1)

# scale by length
roads["traffic_load"] = roads["traffic_base"] * (roads["length_m"] / roads["length_m"].mean())

roads[["highway", "length_m", "traffic_load"]].head()


,highway,length_m,traffic_load
0,residential,2278.588523,24.314052
1,residential,357.548236,3.815277
2,residential,429.094358,4.578722
3,trunk,119.060417,3.387876
4,trunk,109.772074,3.123576


In [70]:
# make sure spatial index is available
import geopandas as gpd

bridges_nearest = gpd.sjoin_nearest(
    bridges,
    roads,
    how="left",
    distance_col="dist_m"
)

bridges_nearest[["bridge_id", "road_id", "dist_m"]].head()


,bridge_id,road_id,dist_m
0,bridge_0,road_21632,0.0
0,bridge_0,road_21634,0.0
0,bridge_0,road_21635,0.0
0,bridge_0,road_21660,0.0
1,bridge_1,road_21639,0.0


In [71]:
bridges_one = (
    bridges_nearest
    .sort_values("dist_m")
    .drop_duplicates(subset="bridge_id")
)

print(len(bridges_one), "bridges after dedup")

bridges_one[["bridge_id", "road_id", "dist_m"]].head()


867 bridges after dedup


,bridge_id,road_id,dist_m
866,bridge_866,road_60542,0.0
862,bridge_862,road_60448,0.0
863,bridge_863,road_60398,0.0
864,bridge_864,road_60398,0.0
865,bridge_865,road_60411,0.0


In [72]:
# small buffer to catch touching roads
roads_buf = roads.copy()
roads_buf["geometry"] = roads_buf.geometry.buffer(1)

road_join = gpd.sjoin(
    roads_buf[["road_id", "geometry"]],
    roads[["road_id", "geometry"]],
    predicate="intersects"
)

# remove self-links
road_edges = road_join[road_join["road_id_left"] != road_join["road_id_right"]]

print("Road-road edges:", len(road_edges))
road_edges.head()


Road-road edges: 185720


,road_id_left,geometry,index_right,road_id_right
0,road_0,"POLYGON ((385618.35 2040356.348, 385618.416 20...",5,road_5
1,road_1,"POLYGON ((385356.923 2040318.544, 385359.811 2...",3,road_3
2,road_2,"POLYGON ((385453.548 2040282.746, 385453.555 2...",4,road_4
2,road_2,"POLYGON ((385453.548 2040282.746, 385453.555 2...",1107,road_1107
2,road_2,"POLYGON ((385453.548 2040282.746, 385453.555 2...",3,road_3


In [73]:
from pathlib import Path

print("CWD:", Path.cwd())
print("Weather raw exists:", Path("data/weather/raw").exists())
print("Files there:", list(Path("data/weather/raw").glob("*")))


CWD: C:\PycharmProjects\EDI\notebooks
Weather raw exists: False
Files there: []


In [74]:
import networkx as nx
import pickle

from risk.flood_risk import attach_rainfall_to_roads

# -------------------------------------------------
# 0) Show current roads columns (debug)
# -------------------------------------------------

print("Original roads columns:")
print(roads.columns)

# -------------------------------------------------
# 1) Compute rainfall + flood risk separately
# -------------------------------------------------

roads_rain = attach_rainfall_to_roads(year=2020)

print("\nRain-only roads columns:")
print(roads_rain.columns)

# -------------------------------------------------
# 2) Remove old weather columns if already present
# -------------------------------------------------

for col in ["rain_mm_mean", "flood_risk"]:
    if col in roads.columns:
        roads = roads.drop(columns=[col])

# -------------------------------------------------
# 3) Merge rainfall into ORIGINAL roads
# -------------------------------------------------

roads = roads.merge(
    roads_rain[["road_id", "rain_mm_mean", "flood_risk"]],
    on="road_id",
    how="left"
)

print("\nMerged roads columns:")
print(roads.columns)

# -------------------------------------------------
# HARD CHECK — stop if merge failed
# -------------------------------------------------

assert "rain_mm_mean" in roads.columns, "rain_mm_mean missing after merge!"
assert "flood_risk" in roads.columns, "flood_risk missing after merge!"

# -------------------------------------------------
# 4) Build graph
# -------------------------------------------------

G = nx.Graph()

for _, r in roads.iterrows():
    G.add_node(
        r["road_id"],
        type="road",
        traffic=r["traffic_load"],
        length=r["length_m"],
        rain_mm_mean=r["rain_mm_mean"],
        flood_risk=r["flood_risk"],
    )

# add bridge nodes
for _, b in bridges_one.iterrows():
    G.add_node(b["bridge_id"], type="bridge")

# road-road edges
for _, e in road_edges.iterrows():
    G.add_edge(
        e["road_id_left"],
        e["road_id_right"],
        kind="road_conn",
    )

# bridge-road edges
for _, r in bridges_one.iterrows():
    G.add_edge(
        r["bridge_id"],
        r["road_id"],
        kind="bridge_on",
    )

print("\nGraph built.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())




Original roads columns:
Index(['road_id', 'highway', 'geometry', 'length_m', 'traffic_base',
       'traffic_load'],
      dtype='object')
Loading rainfall for 2020...
Using rainfall file: RF25_ind2020_rfp25.nc
Loading roads GIS...
Rainfall + flood risk attached to roads.

Rain-only roads columns:
Index(['road_id', 'highway', 'geometry', 'rain_mm_mean', 'flood_risk'], dtype='object')

Merged roads columns:
Index(['road_id', 'highway', 'geometry', 'length_m', 'traffic_base',
       'traffic_load', 'rain_mm_mean', 'flood_risk'],
      dtype='object')

Graph built.
Nodes: 63010
Edges: 93727


In [75]:
from pathlib import Path
import pickle

BASE_DIR = Path.cwd().parents[0]   # notebooks/ -> EDI/

GRAPH_DIR = BASE_DIR / "data" / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

path = GRAPH_DIR / "pune_base_graph_weather.gpickle"

with open(path, "wb") as f:
    pickle.dump(G, f)

print(f"\nSaved -> {path}")



Saved -> C:\PycharmProjects\EDI\data\graphs\pune_base_graph_weather.gpickle


In [76]:
import numpy as np

degrees = [d for _, d in G.degree()]

print("Avg degree:", np.mean(degrees))
print("Max degree:", np.max(degrees))
print("Min degree:", np.min(degrees))


Avg degree: 2.97498809712744
Max degree: 150
Min degree: 0


In [77]:
isolated = list(nx.isolates(G))

print("Isolated nodes:", len(isolated))
isolated[:10]


Isolated nodes: 134


['road_1130',
 'road_1131',
 'road_1132',
 'road_1133',
 'road_3743',
 'road_5640',
 'road_5828',
 'road_6362',
 'road_6396',
 'road_6474']

In [78]:
G.remove_nodes_from(isolated)

print("Nodes after cleanup:", G.number_of_nodes())
print("Edges after cleanup:", G.number_of_edges())


Nodes after cleanup: 62876
Edges after cleanup: 93727


In [79]:
import pickle

with open("../data/graphs/pune_base_graph.gpickle", "wb") as f:
    pickle.dump(G, f)



In [80]:
import pickle

with open("../data/graphs/pune_base_graph.gpickle", "rb") as f:
    G2 = pickle.load(f)

print("Loaded graphs nodes:", G2.number_of_nodes())
print("Loaded graphs edges:", G2.number_of_edges())


Loaded graphs nodes: 62876
Loaded graphs edges: 93727


In [81]:
from shapely.geometry import box
import numpy as np

# bounding box of city
xmin, ymin, xmax, ymax = roads.total_bounds

# create synthetic flood rectangle covering part of city
flood_zone = box(
    xmin,
    ymin,
    xmin + 0.4 * (xmax - xmin),
    ymax
)

# compute flood exposure per road
roads["flood_ratio"] = (
    roads.geometry.intersection(flood_zone).area / roads.geometry.area
)

roads["flood_ratio"] = roads["flood_ratio"].fillna(0)

roads["flood_ratio"].describe()


count    62143.000000
mean         0.000724
std          0.026900
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: flood_ratio, dtype: float64

In [82]:
flooded = roads[roads["flood_ratio"] > 0]

print("Flooded roads:", len(flooded))
print("Percent flooded:", 100 * len(flooded) / len(roads))


Flooded roads: 45
Percent flooded: 0.07241362663534107


In [83]:
# add flood_ratio to graphs nodes
for _, r in roads.iterrows():
    node_id = r["road_id"]
    if node_id in G:
        G.nodes[node_id]["flood_ratio"] = float(r["flood_ratio"])

# bridges: set flood_ratio = 0 for now
for b in bridges_one["bridge_id"]:
    if b in G:
        G.nodes[b]["flood_ratio"] = 0.0

# check few nodes
list(G.nodes(data=True))[:5]


[('road_0',
  {'type': 'road',
   'traffic': 24.314052169681936,
   'length': 2278.588523198561,
   'rain_mm_mean': 3.021176613765499,
   'flood_risk': 0.23843990009159033,
   'flood_ratio': 0.0}),
 ('road_1',
  {'type': 'road',
   'traffic': 3.815277036389211,
   'length': 357.54823619156514,
   'rain_mm_mean': 3.021176613765499,
   'flood_risk': 0.23843990009159033,
   'flood_ratio': 0.0}),
 ('road_2',
  {'type': 'road',
   'traffic': 4.578721650911343,
   'length': 429.09435794074113,
   'rain_mm_mean': 3.021176613765499,
   'flood_risk': 0.23843990009159033,
   'flood_ratio': 0.0}),
 ('road_3',
  {'type': 'road',
   'traffic': 3.3878764946731064,
   'length': 119.06041686670385,
   'rain_mm_mean': 3.021176613765499,
   'flood_risk': 0.23843990009159033,
   'flood_ratio': 0.0}),
 ('road_4',
  {'type': 'road',
   'traffic': 3.1235757245928277,
   'length': 109.77207358930669,
   'rain_mm_mean': 3.021176613765499,
   'flood_risk': 0.23843990009159033,
   'flood_ratio': 0.0})]

In [84]:
# mark failed roads
for n, data in G.nodes(data=True):
    if data.get("type") == "road":
        data["failed"] = data.get("flood_ratio", 0) > 0.3
    else:
        data["failed"] = False

# count failures
failed_nodes = [n for n, d in G.nodes(data=True) if d.get("failed")]

print("Initial failed assets:", len(failed_nodes))
failed_nodes[:10]


Initial failed assets: 32


['road_11393',
 'road_13942',
 'road_17676',
 'road_18326',
 'road_18336',
 'road_18337',
 'road_18340',
 'road_18342',
 'road_18345',
 'road_18351']

In [85]:
import pickle

with open("../data/graphs/pune_flooded_graph.gpickle", "wb") as f:
    pickle.dump(G, f)


In [86]:
# collect failed nodes
failed_nodes = [n for n, d in G.nodes(data=True) if d.get("failed")]

# find neighbors of failed nodes
neighbor_nodes = set()

for n in failed_nodes:
    for nbr in G.neighbors(n):
        neighbor_nodes.add(nbr)

print("Failed nodes:", len(failed_nodes))
print("Direct neighbors:", len(neighbor_nodes))

list(neighbor_nodes)[:10]


Failed nodes: 32
Direct neighbors: 61


['road_18380',
 'road_18600',
 'road_18344',
 'road_18639',
 'road_18617',
 'road_30125',
 'road_18340',
 'road_18326',
 'road_18376',
 'road_18381']

In [87]:
# increase traffic on neighbors
for n in neighbor_nodes:
    if G.nodes[n]["type"] == "road" and not G.nodes[n]["failed"]:
        G.nodes[n]["traffic"] *= 1.5
        G.nodes[n]["overloaded"] = True
    else:
        G.nodes[n]["overloaded"] = False

# check some neighbors
[(n, G.nodes[n]["traffic"], G.nodes[n]["overloaded"]) for n in list(neighbor_nodes)[:10]]


[('road_18380', 0.27214117333408744, False),
 ('road_18600', 0.15292957178000563, True),
 ('road_18344', 0.18020149940223323, True),
 ('road_18639', 0.07924949413581432, True),
 ('road_18617', 2.1099021215680764, False),
 ('road_30125', 0.11131420390435368, True),
 ('road_18340', 1.0089664395323594, False),
 ('road_18326', 2.329479253481196, False),
 ('road_18376', 0.14158165043004367, True),
 ('road_18381', 0.0904812314834611, True)]

In [88]:
new_failures = []

for n, d in G.nodes(data=True):
    if d.get("type") == "road" and not d.get("failed"):
        if d.get("overloaded") and d.get("traffic", 0) > 1.2:
            d["failed"] = True
            new_failures.append(n)

print("New failures:", len(new_failures))
new_failures[:10]


New failures: 4


['road_13798', 'road_18035', 'road_18347', 'road_18611']

In [89]:
second_neighbors = set()

for n in new_failures:
    for nbr in G.neighbors(n):
        second_neighbors.add(nbr)

print("Second-wave neighbors:", len(second_neighbors))
list(second_neighbors)[:10]


Second-wave neighbors: 41


['road_17018',
 'road_13964',
 'road_13923',
 'road_18617',
 'road_18326',
 'road_13965',
 'road_17674',
 'road_18376',
 'road_17378',
 'road_18616']

In [90]:
def cascade_step(G, traffic_threshold=1.2, overload_factor=1.5):
    failed_now = [n for n, d in G.nodes(data=True) if d.get("failed")]

    neighbors = set()
    for n in failed_now:
        for nbr in G.neighbors(n):
            neighbors.add(nbr)

    # overload neighbors
    for n in neighbors:
        if G.nodes[n]["type"] == "road" and not G.nodes[n]["failed"]:
            G.nodes[n]["traffic"] *= overload_factor
            G.nodes[n]["overloaded"] = True

    # trigger new failures
    new_failures = []
    for n, d in G.nodes(data=True):
        if d.get("type") == "road" and not d.get("failed"):
            if d.get("overloaded") and d.get("traffic", 0) > traffic_threshold:
                d["failed"] = True
                new_failures.append(n)

    return new_failures


In [91]:
cascade_history = []

for step in range(10):   # max 10 rounds
    new_failures = cascade_step(G)
    cascade_history.append(len(new_failures))

    print(f"Step {step+1}: new failures = {len(new_failures)}")

    if len(new_failures) == 0:
        print("Cascade stabilized.")
        break

print("Cascade history:", cascade_history)


Step 1: new failures = 8
Step 2: new failures = 20
Step 3: new failures = 63
Step 4: new failures = 135
Step 5: new failures = 238
Step 6: new failures = 342
Step 7: new failures = 413
Step 8: new failures = 543
Step 9: new failures = 623
Step 10: new failures = 702
Cascade history: [8, 20, 63, 135, 238, 342, 413, 543, 623, 702]


In [92]:
total_failed = [n for n, d in G.nodes(data=True) if d.get("failed")]

print("Total failed roads:", len(total_failed))
print("Percent of network:", 100 * len(total_failed) / len(G))


Total failed roads: 3123
Percent of network: 4.96691901520453


In [93]:
# simple repair model
for n, d in G.nodes(data=True):
    if d.get("type") == "road":
        length = d.get("length", 100)

        # days to repair proportional to length
        d["repair_time"] = max(1, length / 200)

        # cost proportional to length
        d["repair_cost"] = length * 500


In [94]:
# simple repair model
for n, d in G.nodes(data=True):
    if d.get("type") == "road":
        length = d.get("length", 100)

        # days to repair proportional to length
        d["repair_time"] = max(1, length / 200)

        # cost proportional to length
        d["repair_cost"] = length * 500


In [95]:
failed_now = [n for n, d in G.nodes(data=True) if d.get("failed")]

print("Currently failed roads:", len(failed_now))



Currently failed roads: 3123


In [96]:
sample = failed_now[0]
sample, G.nodes[sample]


('road_10127',
 {'type': 'road',
  'traffic': 38.63749022839943,
  'length': 905.2269566712372,
  'rain_mm_mean': 3.021176613765499,
  'flood_risk': 0.23843990009159033,
  'flood_ratio': 0.0,
  'failed': True,
  'overloaded': True,
  'repair_time': 4.526134783356186,
  'repair_cost': 452613.4783356186})

In [97]:
priorities = []

for n, d in G.nodes(data=True):
    if d.get("type") == "road" and d.get("failed"):
        deg = G.degree(n)
        traffic = d.get("traffic", 1)

        score = deg * traffic
        priorities.append((n, score))

priorities = sorted(priorities, key=lambda x: x[1], reverse=True)

print("Top 10 priority roads:")
for p in priorities[:10]:
    print(p)


Top 10 priority roads:
('road_18947', 3430.2636143298755)
('road_13553', 2144.486582602187)
('road_10412', 1163.026898912642)
('road_13880', 1142.5622504289752)
('road_11364', 1087.3537670591272)
('road_16249', 825.7759289780272)
('road_16546', 790.0661274872655)
('road_16545', 788.439476922406)
('road_10127', 772.7498045679886)
('road_12758', 760.6893078492375)


In [98]:
# choose top 10 to repair
to_repair = [n for n, _ in priorities[:10]]

for n in to_repair:
    G.nodes[n]["failed"] = False
    G.nodes[n]["overloaded"] = False

print("Repaired roads:", to_repair)


Repaired roads: ['road_18947', 'road_13553', 'road_10412', 'road_13880', 'road_11364', 'road_16249', 'road_16546', 'road_16545', 'road_10127', 'road_12758']


In [99]:
post_history = []

for step in range(10):
    new_failures = cascade_step(G)
    post_history.append(len(new_failures))

    print(f"Post-repair step {step+1}: new failures = {len(new_failures)}")

    if len(new_failures) == 0:
        print("System stabilized after repairs.")
        break

print("Post-repair cascade history:", post_history)


Post-repair step 1: new failures = 625
Post-repair step 2: new failures = 654
Post-repair step 3: new failures = 590
Post-repair step 4: new failures = 624
Post-repair step 5: new failures = 534
Post-repair step 6: new failures = 507
Post-repair step 7: new failures = 569
Post-repair step 8: new failures = 551
Post-repair step 9: new failures = 609
Post-repair step 10: new failures = 576
Post-repair cascade history: [625, 654, 590, 624, 534, 507, 569, 551, 609, 576]


In [100]:
# repair top 200 critical roads
to_repair = [n for n, _ in priorities[:200]]

for n in to_repair:
    G.nodes[n]["failed"] = False
    G.nodes[n]["overloaded"] = False

print("Repaired roads count:", len(to_repair))


Repaired roads count: 200


In [101]:
post200_history = []

for step in range(10):
    new_failures = cascade_step(G)
    post200_history.append(len(new_failures))

    print(f"Post-200 step {step+1}: new failures = {len(new_failures)}")

    if len(new_failures) == 0:
        print("System stabilized after 200 repairs.")
        break

print("Post-200 cascade history:", post200_history)


Post-200 step 1: new failures = 758
Post-200 step 2: new failures = 640
Post-200 step 3: new failures = 706
Post-200 step 4: new failures = 704
Post-200 step 5: new failures = 793
Post-200 step 6: new failures = 795
Post-200 step 7: new failures = 934
Post-200 step 8: new failures = 1017
Post-200 step 9: new failures = 1034
Post-200 step 10: new failures = 1019
Post-200 cascade history: [758, 640, 706, 704, 793, 795, 934, 1017, 1034, 1019]


In [102]:
# assign traffic capacity (simple rule)
for n, d in G.nodes(data=True):
    if d.get("type") == "road":
        # capacity proportional to length
        d["capacity"] = d["traffic"] * 2


In [103]:
def cascade_step_capacity(G, overload_factor=1.3):
    failed_now = [n for n, d in G.nodes(data=True) if d.get("failed")]

    neighbors = set()
    for n in failed_now:
        for nbr in G.neighbors(n):
            neighbors.add(nbr)

    # push traffic to neighbors
    for n in neighbors:
        d = G.nodes[n]
        if d.get("type") == "road" and not d.get("failed"):
            d["traffic"] *= overload_factor

            # check capacity
            if d["traffic"] > d["capacity"]:
                d["overloaded"] = True
            else:
                d["overloaded"] = False

    # trigger failures from overload
    new_failures = []
    for n, d in G.nodes(data=True):
        if d.get("type") == "road" and not d.get("failed"):
            if d.get("overloaded"):
                d["failed"] = True
                new_failures.append(n)

    return new_failures


In [104]:
import pickle

with open("../data/graphs/pune_flooded_graph.gpickle", "rb") as f:
    G = pickle.load(f)

print("Reloaded nodes:", G.number_of_nodes())


Reloaded nodes: 62876


In [105]:
for n, d in G.nodes(data=True):
    if d.get("type") == "road":
        d["capacity"] = d["traffic"] * 2


In [106]:
cap_history = []

for step in range(15):
    new_failures = cascade_step_capacity(G)
    cap_history.append(len(new_failures))

    print(f"Cap step {step+1}: new failures = {len(new_failures)}")

    if len(new_failures) == 0:
        print("Cascade stabilized under capacity rules.")
        break

print("Capacity-based cascade history:", cap_history)


Cap step 1: new failures = 0
Cascade stabilized under capacity rules.
Capacity-based cascade history: [0]


In [107]:
import json

results = {
    "naive_cascade": cascade_history,
    "capacity_cascade": cap_history,
}

with open("../data/graphs/cascade_results.json", "w") as f:
    json.dump(results, f)

print("Saved cascade_results.json")


Saved cascade_results.json
